In [ ]:
!pip install -qU chromadb              # install ChromaDB: a vector database used to store embeddings and perform similarity search
!pip install -qU openai                # install OpenAI client library: used to call LLM APIs (via OpenRouter in our code)
!pip install -qU pypdf2                # install PyPDF2: library for reading and extracting text from PDF files
!pip install -qU python-docx           # install python-docx: library to read/write Microsoft Word (.docx) files
!pip install -qU sentence-transformers # install sentence-transformers: provides pretrained models to convert text into embeddings

In [ ]:
import docx                            # import python-docx module to handle Word documents
import PyPDF2                          # import PyPDF2 module for PDF parsing and text extraction
import os                              # import os module for file path manipulation and system operations

def read_text_file(file_path: str):
    """Read content from a text file"""
    with open(file_path, 'r', encoding='utf-8') as file:  # Open file in read mode with UTF-8 encoding
        return file.read()             # Return full content of the file as a string

def read_pdf_file(file_path: str):
    """Read Content From a PDF File"""
    text=""                           # initialize empty string to store extracted text
    with open(file_path, "rb") as file:  # open file
        pdf_reader=PyPDF2.PdfReader(file)  # create PDF reader object
        for page in pdf_reader.pages:
            text += page.extract_text() + "\n"  # extract text from page and append with newline
    return text                       # return the combined text

def read_docx_file(file_path: str):
    """Read content from a Word document"""
    doc = docx.Document(file_path)   # load the Word document
    return "\n".join([paragraph.text for paragraph in doc.paragraphs])  # extract text from each paragraph and join with newlines


In [ ]:
def read_document(file_path: str):
    """Read Document Content Based on File Extension"""
    _, file_extension = os.path.splitext(file_path)  # split filename and extension
    file_extension = file_extension.lower()         # normalize extension to lowercase
    if file_extension == ".txt":
        return read_text_file(file_path)
    elif file_extension == ".pdf":
        return read_pdf_file(file_path)
    elif file_extension == ".docx":
        return read_docx_file(file_path)
    else:
        raise ValueError(f"Unsupported File Format: {file_extension}")  # Raise error for unsupported formats

In [ ]:
!gdown 1ZVR1vz7_-SQTLu5qlj0kqH46MAo5Bfgn -O /content/GWOELM.pdf  # Download file from Google Drive using gdown

In [ ]:
file_path = "/content/GWOELM.pdf"  # define path to downloaded PDF file
text = read_document(file_path)    # read document content using unified reader

In [ ]:
def split_text(text: str, chunk_size: int = 500, chunk_overlap: int = 100):
    """Split text into overlapping chunks"""
    text = text.replace("\n", " ").strip()  # remove newlines and trim whitespace
    chunks = []                             # initialize list to store chunks
    start = 0                               # start index for chunking
    length = len(text)                      # total length of text

    while start < length:
        end = min(start + chunk_size, length)  # compute end index
        chunk = text[start:end].strip()     # extract chunk and trim whitespace
        if chunk:
            chunks.append(chunk)            # add chunk to list if chuck is not empty
        start += chunk_size - chunk_overlap # Move start forward with overlap

    return chunks                           # return list of chunks

In [ ]:
import chromadb              # import ChromaDB vector database

from chromadb.utils import embedding_functions  # import embedding function utilities

client = chromadb.PersistentClient(path="chroma_db")  # Create persistent database client stored locally

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"         # load pretrained embedding model for semantic encoding. Here you need to provide access token from Hugging Face!
)

collection = client.get_or_create_collection(
    name="documents_collection",          # name of the collection
    embedding_function=sentence_transformer_ef  # attach embedding function to auto-embed documents
)

In [ ]:
def process_document(file_path: str):
    """Process a single document and prepare it for ChromaDB insertion"""
    try:
        content = read_document(file_path)  # Read document content

        chunks = split_text(content)        # Split content into chunks

        file_name = os.path.basename(file_path)  # Extract file name from path
        metadatas = [{"source": file_name, "chunk": i} for i in range(len(chunks))]  # create metadata for each chunk
        ids = [f"{file_name}_chunk_{i}" for i in range(len(chunks))]  # create unique IDs for each chunk

        return ids, chunks, metadatas      # return prepared data
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")  # Print error message
        return [], [], []

In [ ]:
ids, chunks, metadatas = process_document(file_path)  #process document
len(chunks)        # number of chunks

In [ ]:
collection.add(documents=chunks, metadatas=metadatas, ids=ids)  # add chunks and metadata to vector database

In [ ]:
def semantic_search(collection, query: str, n_results: int = 2):
    """Perform a minimal semantic search."""
    return collection.query(                 # query vector DB
        query_texts=[query],                # query text from the user prompt (will be embedded)
        n_results=n_results,                # number of top results to return
        include=["documents", "metadatas"] # include matched documents and metadata
    )

def get_context_with_sources(results):
    """This takes the search results from the previous function and makes them easy to use"""
    if not results or not results.get("documents") or not results["documents"][0]:  # check if results are empty
        return "", []                      # Return empty context and sources

    context = "\n\n".join(results["documents"][0])  # combine retrieved chunks into one context string

    seen = set()                          # track unique sources
    sources = []                          #List to store formatted sources
    for meta in results["metadatas"][0]:
        label = f"{meta.get('source','?')} (chunk {meta.get('chunk','?')})"  # format source label
        if label not in seen:             #avoid duplicates
            seen.add(label)               # mark as seen
            sources.append(label)         # add to sources list

    return context, sources             # return context and sources

In [ ]:
OPEN_ROUTER_API_KEY = "***"  # API key for OpenRouter (LLM gateway). You need the get this from openrouter.ai website.
OPEN_ROUTER_MODEL_NAME = "openai/gpt-4o-mini"  # Model name to use via OpenRouter. There are several free LLM models in the website.

In [ ]:
from openai import OpenAI           # Import OpenAI client

In [ ]:
client = OpenAI(
  base_url = "https://openrouter.ai/api/v1",  # Use OpenRouter endpoint instead of OpenAI default
  api_key = OPEN_ROUTER_API_KEY,              # provide API key
)

SYSTEM_PROMPT = (                      # define system-level instruction for LLM
    "You are a helpful assistant for retrieval-augmented generation (RAG).\n"
    "Answer ONLY using the provided context. "
    "If the answer is not found in the context, say: "
    "'I don't know based on the provided documents.'"
)

def build_messages(context: str, question: str):
    """Create messages for OpenAI API."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},  # system instruction
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"}  # User prompt augmented with context
    ]

In [ ]:
def rag_answer(collection, query: str, n_results: int = 4, model: str = OPEN_ROUTER_MODEL_NAME):
    """Full RAG pipeline: Run semantic search + OpenAI generation, and print results."""
    results = semantic_search(collection, query, n_results)  # retrieve relevant chunks
    context, sources = get_context_with_sources(results)     # build context and sources

    if not context.strip():
        print("No relevant context found.")  # Warn user if no context found
        return "", []

    messages = build_messages(context, query)  # Build LLM input messages

    response = client.chat.completions.create(  # call LLM API
        model=model,                            # specify model
        messages=messages,                      # pass messages
        temperature=0.2,                        # Low randomness for factual answers
        max_tokens=512                          # limit response length
    )

    answer = response.choices[0].message.content.strip()  # extract generated answer

    print("\n=== ANSWER ===\n")     # Print answer header
    print(answer or "[No answer generated]")  # Print answer

    print("\n=== SOURCES ===")      # Print sources header
    if sources:
        for i, s in enumerate(sources, 1):  # Print sources
            print(f"{i}. {s}")
    else:
        print("[No sources found]")  # handle no sources

    return answer, sources

In [ ]:
query = "What is an Extreme Learning Machine?"  # Query example
rag_answer(collection, query, n_results=5)  # Run RAG

In [ ]:
query = "ELM-GWO is compred to which models and how do you rate its performance?"  # Query example
rag_answer(collection, query, n_results=5)      # Run RAG

In [ ]:
query = "How to develope the architecture of ELM?"  # Query example
rag_answer(collection, query, n_results=15)      # Run RAG